# Assignment Module 2: Aircraft Classification

The goal of this assignment is to implement a neural network that classifies images of 100 aircraft model variants from the [Fine-Grained Visual Classification of Aircraft (**FGVC-Aircraft**) dataset](https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/). The assignment is divided into two parts: first, you will be asked to implement your own neural network for image classification from scratch; then, you will fine-tune a pretrained network provided by PyTorch.

![](https:///raw.githubusercontent.com/CVLAB-Unibo/ipcv-assignment-2/master/fgvc_aircraft_variants.svg)

## Part 0 — 环境与配置

环境自动检测（Colab / Kaggle / 本地），设置 `DATA_ROOT`、`DEVICE`、随机种子。

**Kaggle 使用说明**：
- Notebook 设置里要打开 `Settings → Internet: On`（否则下载数据集会失败；第一次开启
  GPU+联网需要给账号做手机号验证）
- 交互式调试用 **Edit** 会话（跟 Colab 类似，也怕断连）；真正要跑几十分钟到几小时、
  不想守着的大批量实验，用 **Save & Run All (Commit)**——整个 notebook 会在 Kaggle
  服务器上后台跑完，跟你的浏览器/电脑完全无关，断网关机都不影响
- checkpoint / 结果存 `/kaggle/working/`，会随这次运行的 Version 输出保存下来

In [ ]:
import os, sys, math, time, json, random, collections
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
from torchvision.datasets import FGVCAircraft
import matplotlib.pyplot as plt

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB:
    DATA_ROOT = Path("/content/data")
elif IN_KAGGLE:
    DATA_ROOT = Path("/kaggle/working/data")
else:
    DATA_ROOT = Path.home() / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()
torch.backends.cudnn.benchmark = True

PLATFORM = "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local"
print(f"torch {torch.__version__} | tv {torchvision.__version__}")
print(f"platform={PLATFORM} | device={DEVICE} | data_root={DATA_ROOT}")
if IN_KAGGLE:
    print("Kaggle 提醒：确认 Notebook Settings -> Internet 已打开，否则下载数据集会失败。")

## Dataset

Download and acces the dataset through its official [PyTorch `FGVCAircraft` class](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.FGVCAircraft.html) (by setting its constructor argument `annotation_level` to `'variant'`).

### 数据集说明

FGVC-Aircraft，100 个 *variant* 类，train/val/test 各 ~3333 张，类别近似均衡。
每张图底部有约 20px 版权条，resize 前先裁掉。
`train` 用于训练、`val` 用于选模型/早停/调参、`test` 只用于报告最终数字。
首次下载约 2.75GB（牛津 VGG 官网，若失败需手动下 tar 包并放到 `DATA_ROOT`）。

In [ ]:
# --- 稳健下载 FGVC-Aircraft（牛津 VGG 官网偶尔 403/超时）---
# torchvision 期望 <DATA_ROOT>/fgvc-aircraft-2013b/ ；若 tar 已存在会跳过下载直接解压。
import urllib.request, shutil, ssl

_TAR = DATA_ROOT / "fgvc-aircraft-2013b.tar.gz"
_URL = "https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/archives/fgvc-aircraft-2013b.tar.gz"

if (DATA_ROOT / "fgvc-aircraft-2013b").exists():
    print("已解压，跳过")
elif _TAR.exists() and _TAR.stat().st_size > 2_000_000_000:
    print(f"tar 已就位 ({_TAR.stat().st_size/1e9:.2f} GB)，交给 torchvision 解压")
else:
    ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    for attempt in range(1, 6):
        try:
            print(f"下载尝试 {attempt}/5 ...")
            req = urllib.request.Request(_URL, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120, context=ctx) as r, open(_TAR, "wb") as f:
                shutil.copyfileobj(r, f, length=1 << 20)
            print(f"完成 {_TAR.stat().st_size/1e9:.2f} GB")
            break
        except Exception as e:
            print("  失败:", e)
    else:
        print("\n自动下载失败。手动方案：把 fgvc-aircraft-2013b.tar.gz 传到\n "
              f"{_TAR}\n然后重跑本 cell 与下一 cell（torchvision 会自动解压）。")

In [ ]:
NUM_CLASSES = 100

def load_raw(split):
    return FGVCAircraft(root=str(DATA_ROOT), split=split,
                        annotation_level="variant", download=True, transform=None)

raw_train, raw_val, raw_test = load_raw("train"), load_raw("val"), load_raw("test")
print(f"train {len(raw_train)} | val {len(raw_val)} | test {len(raw_test)}")
print(f"#classes = {len(raw_train.classes)}  e.g. {raw_train.classes[:4]}")
assert len(raw_train.classes) == NUM_CLASSES

In [ ]:
class CropBottomBanner:
    """去掉 FGVC-Aircraft 底部 ~20px 版权条"""
    def __init__(self, px=20): self.px = px
    def __call__(self, img):
        w, h = img.size
        return img.crop((0, 0, w, h - self.px))

IMG_SIZE  = 128
RESIZE_TO = int(round(IMG_SIZE * 1.14))   # 先放大再 center-crop，保留长宽比

def compute_mean_std(base, n=1500):
    tf = transforms.Compose([CropBottomBanner(20),
                             transforms.Resize(RESIZE_TO),
                             transforms.CenterCrop(IMG_SIZE),
                             transforms.ToTensor()])
    idx = np.random.default_rng(SEED).choice(len(base), min(n, len(base)), replace=False)
    mean = torch.zeros(3); sq = torch.zeros(3)
    for i in idx:
        x = tf(base[i][0]); mean += x.mean((1, 2)); sq += (x**2).mean((1, 2))
    mean /= len(idx); std = (sq/len(idx) - mean**2).sqrt()
    return mean.tolist(), std.tolist()

FGVC_MEAN, FGVC_STD = compute_mean_std(raw_train)
print("mean", [round(v, 4) for v in FGVC_MEAN], "std", [round(v, 4) for v in FGVC_STD])

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def build_train_tf(img_size=IMG_SIZE, mean=None, std=None,
                   augment=True, rrc=True, hflip=True, jitter=True):
    mean = mean or FGVC_MEAN; std = std or FGVC_STD
    steps = [CropBottomBanner(20)]
    if augment and rrc:
        steps.append(transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0), ratio=(0.8, 1.25)))
    else:
        steps += [transforms.Resize(int(round(img_size*1.14))), transforms.CenterCrop(img_size)]
    if augment and hflip:  steps.append(transforms.RandomHorizontalFlip())
    if augment and jitter: steps.append(transforms.ColorJitter(0.2, 0.2, 0.2))
    steps += [transforms.ToTensor(), transforms.Normalize(mean, std)]
    return transforms.Compose(steps)

def build_eval_tf(img_size=IMG_SIZE, mean=None, std=None):
    mean = mean or FGVC_MEAN; std = std or FGVC_STD
    return transforms.Compose([CropBottomBanner(20),
                               transforms.Resize(int(round(img_size*1.14))),
                               transforms.CenterCrop(img_size),
                               transforms.ToTensor(), transforms.Normalize(mean, std)])

In [ ]:
class Transformed(Dataset):
    def __init__(self, base, tf): self.base, self.tf = base, tf
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        img, y = self.base[i]; return self.tf(img), y

def make_loaders(img_size=IMG_SIZE, batch_size=128, mean=None, std=None,
                 augment=True, rrc=True, hflip=True, jitter=True, num_workers=2):
    ttf = build_train_tf(img_size, mean, std, augment, rrc, hflip, jitter)
    etf = build_eval_tf(img_size, mean, std)
    common = dict(num_workers=num_workers, pin_memory=(DEVICE.type == "cuda"))
    return (DataLoader(Transformed(raw_train, ttf), batch_size, shuffle=True,  drop_last=True, **common),
            DataLoader(Transformed(raw_val,   etf), batch_size, shuffle=False,                 **common),
            DataLoader(Transformed(raw_test,  etf), batch_size, shuffle=False,                 **common))

## EDA

In [ ]:
def denorm(x, mean=None, std=None):
    mean = torch.tensor(mean or FGVC_MEAN).view(3, 1, 1)
    std  = torch.tensor(std  or FGVC_STD ).view(3, 1, 1)
    return (x*std + mean).clamp(0, 1)

# 1) 增强后样本网格
tl, _, _ = make_loaders(batch_size=16, num_workers=0)
xb, yb = next(iter(tl))
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, im, y in zip(axes.flat, xb, yb):
    ax.imshow(denorm(im).permute(1, 2, 0).numpy()); ax.axis("off")
    ax.set_title(raw_train.classes[y][:14], fontsize=7)
plt.suptitle("augmented training samples"); plt.tight_layout(); plt.show()

# 2) 类别分布
_lbl = getattr(raw_train, "_labels", None)
lbl = np.array(_lbl if _lbl is not None else [y for _, y in raw_train])
cnt = np.bincount(lbl, minlength=NUM_CLASSES)
plt.figure(figsize=(10, 2.5)); plt.bar(range(NUM_CLASSES), cnt)
plt.title(f"train imgs/class  (min {cnt.min()}, max {cnt.max()})"); plt.show()

# 3) 原图尺寸
s = np.array([raw_train[i][0].size for i in np.random.default_rng(0).integers(0, len(raw_train), 200)])
print(f"raw W {s[:,0].min()}-{s[:,0].max()} | raw H {s[:,1].min()}-{s[:,1].max()}")

## Part 1: design your own network

Your goal is to implement a convolutional neural network for image classification and train it from scratch on `FGVCAircraft`. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~50%. You are free to achieve this however you want, except for a few rules you must follow:

- Compile this notebook by displaying the results obtained by the best model you found throughout your experimentation; then show how, by removing some of its components, its performance drops. In other words, do an *ablation study* to prove that your design choices have a positive impact on the final result.

- Do not instantiate an off-the-self PyTorch network. Instead, construct your network as a composition of existing PyTorch layers. In more concrete terms, you can use e.g. `torch.nn.Linear`, but you cannot use e.g. `torchvision.models.alexnet`.

- Show your results and ablations with plots, tables, images, etc. — the clearer, the better.

Don't be too concerned with your model performance: the ~50% is just to give you an idea of when to stop. Keep in mind that a thoroughly justified model with lower accuracy will be rewarded more points than a poorly experimentally validated model with higher accuracy.

### 1.1  架构：ResNet-lite（可配置）

`stem`(stride-2) + 4 个 stage，每个 stage 首个 block 做 stride-2 下采样：
`128 → 64 → 32 → 16 → 8 → 4`，最后 `512×4×4` 特征图 → **全局平均池化** → Dropout → `Linear(512, 100)`。

- 每个 `BasicBlock` = 两层 `Conv3×3 →(BN)→ 激活`；`use_residual=True` 时加 shortcut，
  尺寸/通道变化处用 `1×1 conv` 投影（ResNet 标准做法）。
- Kaiming (`fan_out`) 初始化卷积，BN 权重初始化为 1、偏置为 0。
- **所有设计选择都是 `NetConfig` 的一个开关**，消融时只改配置：
  `use_bn` · `use_residual` · `activation` · `global_pool`(avg/flatten) · `dropout` · `stage_channels`(容量)。

In [ ]:
from dataclasses import dataclass, asdict

ACT = {"relu": nn.ReLU, "gelu": nn.GELU, "leaky_relu": lambda: nn.LeakyReLU(0.1)}

@dataclass
class NetConfig:
    num_classes:      int   = 100
    stem_channels:    int   = 32
    stage_channels:   tuple = (64, 128, 256, 512)
    blocks_per_stage: int   = 2
    use_bn:           bool  = True         # 消融
    use_residual:     bool  = True         # 消融
    activation:       str   = "relu"       # relu | gelu | leaky_relu   消融
    global_pool:      str   = "avg"        # "avg" | "flatten"          消融
    dropout:          float = 0.3          # 消融
    in_size:          int   = 128

def _norm(c, use_bn):
    return nn.BatchNorm2d(c) if use_bn else nn.Identity()

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride, cfg):
        super().__init__()
        act = ACT[cfg.activation]
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=not cfg.use_bn)
        self.n1, self.act1 = _norm(cout, cfg.use_bn), act()
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=not cfg.use_bn)
        self.n2, self.act2 = _norm(cout, cfg.use_bn), act()
        self.use_residual = cfg.use_residual
        if cfg.use_residual and (stride != 1 or cin != cout):
            self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=not cfg.use_bn),
                                       _norm(cout, cfg.use_bn))
        else:
            self.short = nn.Identity()

    def forward(self, x):
        out = self.act1(self.n1(self.conv1(x)))
        out = self.n2(self.conv2(out))
        if self.use_residual:
            out = out + self.short(x)
        return self.act2(out)

class AircraftNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        act = ACT[cfg.activation]
        self.stem = nn.Sequential(
            nn.Conv2d(3, cfg.stem_channels, 3, 2, 1, bias=not cfg.use_bn),
            _norm(cfg.stem_channels, cfg.use_bn), act())
        blocks, cin = [], cfg.stem_channels
        for cout in cfg.stage_channels:
            for bi in range(cfg.blocks_per_stage):
                blocks.append(BasicBlock(cin, cout, 2 if bi == 0 else 1, cfg))
                cin = cout
        self.stages = nn.Sequential(*blocks)
        feat = cfg.stage_channels[-1]
        if cfg.global_pool == "avg":
            self.pool, head_in = nn.AdaptiveAvgPool2d(1), feat
        elif cfg.global_pool == "flatten":
            r = cfg.in_size // 2 // (2 ** len(cfg.stage_channels))
            self.pool, head_in = nn.Identity(), feat * r * r
        else:
            raise ValueError(cfg.global_pool)
        self.drop = nn.Dropout(cfg.dropout)
        self.fc   = nn.Linear(head_in, cfg.num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x); x = self.stages(x); x = self.pool(x)
        x = torch.flatten(x, 1); x = self.drop(x)
        return self.fc(x)

def build_model(cfg=None):
    return AircraftNet(cfg or NetConfig())

_cfg = NetConfig(); _m = build_model(_cfg)
print(f"params: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M | "
      f"out: {tuple(_m(torch.randn(2, 3, _cfg.in_size, _cfg.in_size)).shape)}")
del _cfg, _m

### 1.2  训练框架（Part 1 / Part 2 共用）

- `TrainConfig` 汇总所有训练超参；`build_optimizer` 支持 SGD / AdamW，`build_scheduler`
  支持 `cosine`（线性 warmup + 余弦退火）与 `constant`（平 LR，用于"去掉调度"消融）。
- `train_loop`：CUDA 上开 AMP 混合精度，每个 epoch 记录 `lr / train_loss / train_top1 /
  val_loss / val_top1 / val_top5`，按 **val top-1** 保存 best 权重。
- `run_experiment(name, net_cfg, train_cfg)`：建 loader → 建模型 → 训练 → 用 best 权重在
  **test** 上评估（top-1 + top-5），登记进全局 `RESULTS`，并把 checkpoint 存到 `CKPT_DIR`
  （Colab 挂 Drive），结果 dump 成 `results.json`（断线不丢）。
- 划分严格：`train` 训练、`val` 选模型、`test` 只在最后报一次。

In [ ]:
# checkpoint / 结果目录：
#   Colab  -> 挂 Google Drive（断线也不丢）
#   Kaggle -> /kaggle/working（随 Version 输出保存；Save & Run All 后台跑完也不会丢）
#   本地   -> ./ckpts
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = Path("/content/drive/MyDrive/ipcv_assignment2/ckpts")
elif IN_KAGGLE:
    CKPT_DIR = Path("/kaggle/working/ckpts")
else:
    CKPT_DIR = Path("ckpts")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_JSON = CKPT_DIR.parent / "results.json"
print("CKPT_DIR   :", CKPT_DIR)
print("RESULTS_JSON:", RESULTS_JSON)

In [ ]:
from dataclasses import dataclass, field, asdict, replace

@dataclass
class TrainConfig:
    epochs:          int   = 80
    batch_size:      int   = 128
    optimizer:       str   = "sgd"       # sgd | adamw
    lr:              float = 0.1         # adamw 时建议 ~3e-4
    momentum:        float = 0.9
    weight_decay:    float = 5e-4
    nesterov:        bool  = True
    scheduler:       str   = "cosine"    # cosine | constant   ← 消融
    warmup_epochs:   int   = 5
    label_smoothing: float = 0.0         # 2B 调参用
    grad_clip:       float = 0.0
    amp:             bool  = True        # 仅 CUDA 生效
    img_size:        int   = 128
    mean: tuple = None                   # None→FGVC 自算; Part 2 传 IMAGENET_MEAN
    std:  tuple = None
    augment: bool = True
    rrc:     bool = True
    hflip:   bool = True
    jitter:  bool = True
    num_workers: int = 2
    seed: int = 42


def build_optimizer(model, cfg: TrainConfig):
    if cfg.optimizer == "sgd":
        return torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum,
                               weight_decay=cfg.weight_decay, nesterov=cfg.nesterov)
    if cfg.optimizer == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    raise ValueError(cfg.optimizer)


def build_scheduler(opt, cfg: TrainConfig):
    """按 epoch 步进。constant = 平 LR（去掉调度的消融）。"""
    if cfg.scheduler == "constant":
        return torch.optim.lr_scheduler.LambdaLR(opt, lambda _: 1.0)
    warm = torch.optim.lr_scheduler.LinearLR(
        opt, start_factor=1e-2, total_iters=max(1, cfg.warmup_epochs))
    cos = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max(1, cfg.epochs - cfg.warmup_epochs))
    return torch.optim.lr_scheduler.SequentialLR(
        opt, [warm, cos], milestones=[max(1, cfg.warmup_epochs)])

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    tot = 0; loss_sum = 0.0; c1 = 0; c5 = 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        out = model(x)
        loss_sum += criterion(out, y).item() * x.size(0); tot += x.size(0)
        top5 = out.topk(5, dim=1).indices
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
    return loss_sum / tot, 100 * c1 / tot, 100 * c5 / tot


def train_loop(model, train_loader, val_loader, cfg: TrainConfig, verbose=True):
    crit  = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    opt   = build_optimizer(model, cfg)
    sched = build_scheduler(opt, cfg)
    use_amp = cfg.amp and DEVICE.type == "cuda"
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

    keys = ["lr", "train_loss", "train_top1", "val_loss", "val_top1", "val_top5"]
    hist = {k: [] for k in keys}
    best_val = -1.0; best_state = None

    for ep in range(1, cfg.epochs + 1):
        model.train()
        t0 = time.time(); seen = 0; run_loss = 0.0; run_c1 = 0
        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward()
            if cfg.grad_clip > 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update()
            run_loss += loss.item() * x.size(0); seen += x.size(0)
            run_c1   += (out.argmax(1) == y).sum().item()
        sched.step()

        tr_loss = run_loss / seen; tr_top1 = 100 * run_c1 / seen
        va_loss, va_top1, va_top5 = evaluate(model, val_loader, crit)
        lr_now = opt.param_groups[0]["lr"]
        for k, v in zip(keys, [lr_now, tr_loss, tr_top1, va_loss, va_top1, va_top5]):
            hist[k].append(v)
        if va_top1 > best_val:
            best_val = va_top1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"ep {ep:3d}/{cfg.epochs} | lr {lr_now:.4f} | "
                  f"train {tr_loss:.3f}/{tr_top1:5.1f}% | "
                  f"val {va_loss:.3f}/{va_top1:5.1f}% (top5 {va_top5:4.1f}%) | "
                  f"{time.time() - t0:4.1f}s")
    return hist, best_state, best_val

In [ ]:
RESULTS = {}

def _dump_results():
    slim = {k: {kk: vv for kk, vv in v.items() if kk != "history"} for k, v in RESULTS.items()}
    RESULTS_JSON.write_text(json.dumps({"slim": slim, "full": RESULTS}, indent=2, default=str))

def run_experiment(name, net_cfg=None, train_cfg=None, save_ckpt=True, verbose=True):
    net_cfg   = net_cfg   or NetConfig()
    train_cfg = train_cfg or TrainConfig()
    net_cfg   = replace(net_cfg, in_size=train_cfg.img_size)      # 保持一致
    seed_everything(train_cfg.seed)
    tl, vl, testl = make_loaders(
        img_size=train_cfg.img_size, batch_size=train_cfg.batch_size,
        mean=train_cfg.mean, std=train_cfg.std,
        augment=train_cfg.augment, rrc=train_cfg.rrc,
        hflip=train_cfg.hflip, jitter=train_cfg.jitter, num_workers=train_cfg.num_workers)
    model = build_model(net_cfg).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"=== {name} === {n_params/1e6:.2f}M params | img {train_cfg.img_size} | "
          f"{train_cfg.optimizer} lr{train_cfg.lr} wd{train_cfg.weight_decay} | "
          f"sched={train_cfg.scheduler} | {train_cfg.epochs} ep")
    t0 = time.time()
    hist, best_state, best_val = train_loop(model, tl, vl, train_cfg, verbose=verbose)
    model.load_state_dict(best_state)
    _, te1, te5 = evaluate(model, testl, nn.CrossEntropyLoss())
    rec = dict(name=name, test_top1=te1, test_top5=te5, best_val_top1=best_val,
               n_params=int(n_params), minutes=(time.time() - t0) / 60,
               history=hist, net_cfg=asdict(net_cfg), train_cfg=asdict(train_cfg))
    RESULTS[name] = rec
    if save_ckpt:
        torch.save({"model": best_state, "net_cfg": asdict(net_cfg),
                    "train_cfg": asdict(train_cfg), "test_top1": te1},
                   CKPT_DIR / f"{name}.pt")
    _dump_results()
    print(f"[{name}]  test top1 {te1:.2f}%  top5 {te5:.2f}%  |  best val {best_val:.2f}%  "
          f"|  {rec['minutes']:.1f} min")
    return rec


def plot_history(hist_or_name, title=""):
    h = RESULTS[hist_or_name]["history"] if isinstance(hist_or_name, str) else hist_or_name
    ep = range(1, len(h["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    ax[0].plot(ep, h["train_loss"], label="train"); ax[0].plot(ep, h["val_loss"], label="val")
    ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(ep, h["train_top1"], label="train"); ax[1].plot(ep, h["val_top1"], label="val")
    ax[1].set_title("top-1 (%)"); ax[1].set_xlabel("epoch"); ax[1].legend()
    ax[2].plot(ep, h["lr"]); ax[2].set_title("lr"); ax[2].set_xlabel("epoch")
    fig.suptitle(title or (hist_or_name if isinstance(hist_or_name, str) else ""))
    plt.tight_layout(); plt.show()


def results_table(names=None):
    import pandas as pd
    names = names or list(RESULTS)
    rows = [dict(experiment=n,
                 test_top1=round(RESULTS[n]["test_top1"], 2),
                 test_top5=round(RESULTS[n]["test_top5"], 2),
                 val_top1=round(RESULTS[n]["best_val_top1"], 2),
                 params_M=round(RESULTS[n]["n_params"] / 1e6, 2),
                 min=round(RESULTS[n]["minutes"], 1)) for n in names]
    return pd.DataFrame(rows).set_index("experiment")

### 1.3  训练最佳模型与超参探索

**策略**：先跑一个 baseline，观察真实训练曲线（train vs val 差距、饱和 epoch、是否震荡），
再据此决定后续搜索方向，而不是盲目大网格搜索。

**Baseline** = 默认 `NetConfig()` + `TrainConfig()`：
ResNet-lite ~9–10M 参数 · 128px · 全增强 · BN + 残差 + dropout 0.3 ·
SGD lr 0.1 + 5 ep warmup + cosine · wd 5e-4 · batch 128 · 80 epoch。

后续候选（看 baseline 结果再挑）：模型容量 `(48,96,192,384)` / `blocks_per_stage=1`；
学习率 {0.05, 0.1, 0.2}；AdamW lr 3e-4 对照；epoch 80→120；dropout {0.2,0.3,0.5}；分辨率 160。
兜底：MixUp/CutMix、RandAugment、更高分辨率、TTA。

In [ ]:
# --- Phase 4 baseline：默认配置 ---
base_net   = NetConfig()
base_train = TrainConfig(epochs=80)
run_experiment("p1_baseline", net_cfg=base_net, train_cfg=base_train)

In [ ]:
results_table()

In [ ]:
plot_history("p1_baseline", title="p1_baseline — SGD lr0.1 cosine, 128px, full aug")

#### baseline 结果分析

- **最终结果**：test top-1 **34.65%**，test top-5 **63.43%**，best val top-1 **34.77%**
  （恰好出现在最后一个 epoch，80/80），11.2M 参数，训练耗时 111 分钟。
- **严重过拟合**：`train_top1` 在 epoch ~55 冲到接近 99% 并封顶；`val_top1` 从 epoch ~60 起
  基本卡在 33–35% 的窄带里反复小幅震荡——train/val 差距接近 **65 个百分点**。
- **val loss 仍在缓慢下降**（峰值 ~4.3 → 收尾 ~3.35），但 `val_top1` 几乎不再涨：模型对
  已经答对的样本越来越自信，但没能学会答对更多新样本。
- **不是训练时长的问题**：最优 val 恰好落在最后一个 epoch，且最后 ~20 个 epoch 曲线已经
  很平——瓶颈在于**模型容量（11.2M 参数）相对 3334 张训练图明显过剩**，当前正则化
  （BN + 数据增强 + dropout 0.3 + wd 5e-4）压不住。
- 距目标 test top-1 ≈ 50% 还差 **~15 个百分点**。

**结论**：下一步的探针实验（缩容量 / 缩容量+强正则 / AdamW）方向正确，重点看哪组能真正
收窄 train/val gap，并在同样 35 epoch 下拿到更高的 val top-1。

#### 探针实验（35 epoch，快速筛方向）

baseline（11.2M 参数）在 epoch 30 就出现 train 72% / val 24% 的大幅过拟合，且 val 震荡。
80 epoch 太贵（~110 min/组），先用 35 epoch 的短跑筛出方向，赢家再跑满做 `p1_best`。

- `p1_probe_smallcap`：缩小模型容量 `(48,96,192,384)`，其余同 baseline
- `p1_probe_smallcap_reg`：缩容量 + weight_decay ×2 + dropout 0.5
- `p1_probe_adamw`：baseline 容量，换 AdamW（lr 3e-4, wd 1e-2）

In [ ]:
# --- 探针 1：缩小模型容量 ---
run_experiment(
    "p1_probe_smallcap",
    net_cfg=NetConfig(stage_channels=(48, 96, 192, 384)),
    train_cfg=TrainConfig(epochs=35),
)

In [ ]:
# --- 探针 2：缩容量 + 更强正则 ---
run_experiment(
    "p1_probe_smallcap_reg",
    net_cfg=NetConfig(stage_channels=(48, 96, 192, 384), dropout=0.5),
    train_cfg=TrainConfig(epochs=35, weight_decay=1e-3),
)

In [ ]:
# --- 探针 3：AdamW 对照（baseline 容量）---
run_experiment(
    "p1_probe_adamw",
    net_cfg=NetConfig(),
    train_cfg=TrainConfig(epochs=35, optimizer="adamw", lr=3e-4, weight_decay=1e-2),
)

In [ ]:
results_table()

#### 探针实验结果分析

| 实验 | 参数量 | epoch | test top1 | test top5 | val top1 | train top1(末) | train-val 差 |
|---|---|---|---|---|---|---|---|
| p1_baseline | 11.2M | 80 | 34.65% | 63.43% | 34.77% | 99.0% | 64.3 |
| p1_probe_smallcap | 6.32M | 35 | 31.14% | 63.37% | 31.68% | 86.6% | 55.0 |
| **p1_probe_smallcap_reg** | 6.32M | 35 | **31.89%** | **66.16%** | 31.86% | 72.9% | **41.0** |
| p1_probe_adamw | 11.2M | 35 | 19.56% | 42.00% | 18.84% | 78.1% | 59.3 |

- **AdamW 组明显最差**：35 epoch 内收敛比 SGD 慢很多（train 只到 78%），test top1 只有
  19.56%，本次配置下放弃这个方向。
- **缩容量确实压住了过拟合**：train/val 差距 baseline 64.3 → smallcap 55.0 →
  smallcap_reg **41.0**，容量减半 + 加强正则（dropout 0.5、wd 1e-3）效果叠加，方向正确。
- **`p1_probe_smallcap_reg` 最健康**：差距最小、top5 全场最高，且 35 epoch 结束时
  **还没饱和**（最后 5 个 epoch val 仍从 30.1% 稳步涨到 31.9%），不像其他组已经走平。
- baseline 的经验是：进入余弦退火尾声（LR→0 的最后 ~20 epoch），val 会有一次干净的
  上涨（这里从 ~27% 涨到 34.8%，即"退火尾部红利"）。`smallcap_reg` 目前只训了 35 epoch，
  尚未吃到"训满 80 epoch"该有的尾部红利，因此把它训满是下一步的合理选择。

**结论**：以 `smallcap_reg` 的配置（容量减半、dropout 0.5、wd 1e-3）训满 80 epoch，
作为 `p1_best` 候选。

In [ ]:
# --- p1_best 候选：缩容量 + 更强正则，训满 80 epoch ---
run_experiment(
    "p1_best_v1",
    net_cfg=NetConfig(stage_channels=(48, 96, 192, 384), dropout=0.5),
    train_cfg=TrainConfig(epochs=80, weight_decay=1e-3),
)

In [ ]:
results_table()

In [ ]:
plot_history("p1_best_v1", title="p1_best_v1 — SGD lr0.1 cosine, 6.32M params, dropout0.5 wd1e-3, 80ep")

#### p1_best_v1 结果分析

| | test top1 | test top5 | best val | 参数量 | 耗时 |
|---|---|---|---|---|---|
| p1_baseline | 34.65% | 63.43% | 34.77% | 11.2M | 111.1min |
| **p1_best_v1** | **42.36%** | **74.08%** | 41.19% | 6.32M | 57.7min |
| 提升 | **+7.71** | **+10.65** | +6.42 | 减半 | 减半 |

- **参数减半、耗时减半，效果反而大幅提升**——容量匹配数据规模 + 加强正则化
  （dropout 0.5、wd 1e-3）比单纯堆参数更有效。
- **"退火尾部红利"再次出现**：epoch 57 时 val 才 37.4%，到 epoch 80 稳步爬到 41.2%，
  最后 23 个 epoch 贡献了近 4 个点，跟 baseline 的模式一致。
- **过拟合缺口收窄但未消除**：train/val 差距从 baseline 的 64.3 降到 **57.5**
  （末尾 train 98.7% vs val 41.2%），比 baseline 健康很多，但仍是主导因素。
- 最后 4 个 epoch（77-80：41.0/40.9/40.8/41.2%）已趋于平台，单纯再堆 epoch 数收益有限。
- 距 ~50% 目标还差 ~8 个点。下一步：加 `label_smoothing=0.1`（低成本、经典正则化）
  再跑一组，看能否进一步逼近目标。

#### 为什么试 label smoothing

课程 P2L6（Regularization and Training Recipes）里的论证：交叉熵损失永远不会"满意"——
softmax 比值 `correct / (correct + others)` 只有在正确类 logit → +∞，或其余类 logit → -∞
时才能等于 1（loss=0）。由于网络永远达不到无穷，即使样本已经分类正确，loss 仍会持续把
正确类 logit 往上推、其余类往下压——这正是导致训练集 accuracy 冲到 99% 而 val 停滞不前
（`p1_best_v1` 的过拟合缺口）的机制之一。

**Label smoothing**（Inception 论文提出）把 one-hot 目标换成软化目标：正确类目标设为
`1-epsilon`，其余每类分到 `epsilon/C`。这样目标本身就是有限值，loss 在有限 logit 处就能
"满意"，不再无止境地把权重推向极端——本质上是另一种正则化手段。PyTorch v1.10+ 起
`nn.CrossEntropyLoss(label_smoothing=...)` 原生支持；老师建议的合理取值范围是 **10%~15%**，
我们用 `0.1`，在建议范围内。

In [ ]:
# --- p1_best 候选 v2：同容量+正则，再加 label smoothing ---
run_experiment(
    "p1_best_v2",
    net_cfg=NetConfig(stage_channels=(48, 96, 192, 384), dropout=0.5),
    train_cfg=TrainConfig(epochs=80, weight_decay=1e-3, label_smoothing=0.1),
)

In [ ]:
results_table()

In [ ]:
plot_history("p1_best_v2", title="p1_best_v2 — SGD lr0.1 cosine, 6.32M params, dropout0.5 wd1e-3 label_smoothing0.1, 80ep")

#### p1_best_v2 结果分析

| | test top1 | test top5 | val top1 | train top1(末) |
|---|---|---|---|---|
| p1_best_v1（无 label smoothing） | **42.36%** | **74.08%** | 41.19% | 98.7% |
| p1_best_v2（label_smoothing=0.1） | 40.89% | 70.18% | 40.92% | ~99.1% |
| 差异 | -1.47 | -3.90 | -0.27（噪声内） | 基本没变 |

- **label smoothing 没有帮上忙，反而小幅拖累**——test top1 -1.47，test top5 -3.90。
- **关键线索**：两组 train top1 几乎一样高（98.7% vs 99.1%），说明 label smoothing
  **没有真正缓解过拟合**。原因在于它只改变目标概率的形状（正确类目标从 1 降到 0.9），
  但 top-1 accuracy 只看 argmax 是否正确，不管概率具体多高——所以它没有触及"记住训练集
  哪个类是对的"这个根本机制，跟缩容量/dropout 从模型能力上限制过拟合完全不同。
- top5 掉得比 top1 更多，可能是因为 label smoothing 把所有错误类的目标概率拉平到同一
  水平，模糊了"次优选项"之间的相对排序，而 top5 恰恰依赖这个排序（观察到的现象，
  非绝对定论）。
- **结论：不采用 label smoothing，最终 `p1_best` = `p1_best_v1`**
  （test top1 42.36% / test top5 74.08%，6.32M 参数）。这次尝试本身是一次基于 P2L6
  课堂内容的合理假设检验，结果是负面的，同样值得写入报告。

### 1.4  消融实验（Ablation Study）

以 `p1_best`（`p1_best_v1`：容量 6.32M、dropout 0.5、wd 1e-3、80 epoch）为基准，
**每次只翻转一个设计选择**，其余全部保持不变，逐一证明每个组件的贡献：

| 消融项 | 改法 | 预期 |
|---|---|---|
| `p1_abl_no_bn` | `use_bn=False` | lr=0.1 是按"有 BN"调的，去掉后可能训练不稳定/发散 |
| `p1_abl_no_residual` | `use_residual=False` | 收敛变慢、更难训 |
| `p1_abl_no_dropout` | `dropout=0.0` | train/val 差距重新拉大 |
| `p1_abl_flatten_head` | `global_pool="flatten"` | 分类头参数 5万→82万，过拟合更明显 |
| `p1_abl_no_augment` | `augment=False` | 没有数据增强，过拟合更严重 |
| `p1_abl_no_schedule` | `scheduler="constant"` | 没有余弦退火，丧失"尾部红利"，精度更低更震荡 |

全部用跟 `p1_best` 完全相同的 80 epoch 预算，保证对比公平。

In [ ]:
# --- 消融 1：去掉 BatchNorm ---
run_experiment(
    "p1_abl_no_bn",
    net_cfg=NetConfig(stage_channels=(48, 96, 192, 384), dropout=0.5, use_bn=False),
    train_cfg=TrainConfig(epochs=80, weight_decay=1e-3),
)

In [ ]:
# --- 消融 2：去掉残差连接 ---
run_experiment(
    "p1_abl_no_residual",
    net_cfg=NetConfig(stage_channels=(48, 96, 192, 384), dropout=0.5, use_residual=False),
    train_cfg=TrainConfig(epochs=80, weight_decay=1e-3),
)

In [ ]:
# --- 消融 3：去掉 Dropout ---
run_experiment(
    "p1_abl_no_dropout",
    net_cfg=NetConfig(stage_channels=(48, 96, 192, 384), dropout=0.0),
    train_cfg=TrainConfig(epochs=80, weight_decay=1e-3),
)

In [ ]:
# --- 消融 4：GAP 换成 flatten ---
run_experiment(
    "p1_abl_flatten_head",
    net_cfg=NetConfig(stage_channels=(48, 96, 192, 384), dropout=0.5, global_pool="flatten"),
    train_cfg=TrainConfig(epochs=80, weight_decay=1e-3),
)

In [ ]:
# --- 消融 5：去掉数据增强 ---
run_experiment(
    "p1_abl_no_augment",
    net_cfg=NetConfig(stage_channels=(48, 96, 192, 384), dropout=0.5),
    train_cfg=TrainConfig(epochs=80, weight_decay=1e-3, augment=False),
)

In [ ]:
# --- 消融 6：去掉 LR 调度（平学习率）---
run_experiment(
    "p1_abl_no_schedule",
    net_cfg=NetConfig(stage_channels=(48, 96, 192, 384), dropout=0.5),
    train_cfg=TrainConfig(epochs=80, weight_decay=1e-3, scheduler="constant"),
)

In [ ]:
ABLATION_NAMES = ["p1_best_v1", "p1_abl_no_bn", "p1_abl_no_residual", "p1_abl_no_dropout",
                  "p1_abl_flatten_head", "p1_abl_no_augment", "p1_abl_no_schedule"]
tbl = results_table(ABLATION_NAMES)
display(tbl)

fig, ax = plt.subplots(figsize=(10, 4))
vals = [RESULTS[n]["test_top1"] for n in ABLATION_NAMES]
colors_ = ["#2f5da8"] + ["#c0392b"] * (len(ABLATION_NAMES) - 1)
ax.bar(range(len(ABLATION_NAMES)), vals, color=colors_)
ax.set_xticks(range(len(ABLATION_NAMES)))
ax.set_xticklabels(["best\n(baseline)"] + [n.replace("p1_abl_", "-") for n in ABLATION_NAMES[1:]],
                   rotation=20, ha="right")
ax.set_ylabel("test top-1 (%)")
ax.axhline(vals[0], color="#2f5da8", linestyle="--", linewidth=1, alpha=0.6)
ax.set_title("Ablation study — test top-1 relative to p1_best")
plt.tight_layout(); plt.show()

#### 消融实验结果分析

> _待填_：6 组跑完后把 `results_table()` 和柱状图贴回，逐项对照预期填写：
> - 哪个组件去掉后掉分最多？跟预期一致吗？
> - `p1_abl_no_bn` 训练是否真的不稳定（看 loss 曲线是否震荡/发散）？
> - `p1_abl_no_schedule` 是否验证了"退火尾部红利"的说法（对比 p1_best 最后 20 epoch 的涨幅）？
> - 综合这 6 组，给出"每个设计选择贡献了多少"的整体结论，作为 Part 1 的收尾。

## Part 2: fine-tune an existing network

Your goal is to fine-tune a pretrained ResNet-18 model on `FGVCAircraft`. Use the implementation provided by PyTorch, i.e. the opposite of part 1. Specifically, use the PyTorch ResNet-18 model pretrained on ImageNet-1K (V1). Divide your fine-tuning into two parts:

2A. First, fine-tune the ResNet-18 with the same training hyperparameters you used for your best model in part 1.

2B. Then, tweak the training hyperparameters to increase the accuracy on the test split. Justify your choices by analyzing the training plots and/or citing sources that guided you in your decisions — papers, blog posts, YouTube videos, or whatever else you may find useful. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~70%.